# Stitching together `.cha` files from the checkpoints

Perhaps inadvertently, I've saved the outputs for each conversation as a CEDA checkpoint. We need to iterate through them and stitch together the final file for analysis.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
DATA_PATH = 'data'

INPUT_PATH = os.path.join(DATA_PATH, 'chas-ckpts')

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

OUTPUT_NAME = os.path.join(OUTPUT_PATH,'cha-MICASE-ceda-analysis.tsv')

In [ ]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

## Grabbing each file and creating a merged dataframe

In [ ]:
files = [f for f in grab_all_files(DATA_PATH, '4.csv.pt') if (not f.startswith('.')) and ('cha_corpus' in f)]
print(files)

In [ ]:
df_ = []
for f in tqdm(files):
    ckpt = torch.load(f)
    df = pd.DataFrame(ckpt['meta_data'])
    
    # number of tokens per text
    df['nx'] = ckpt['N'][:,0].tolist()
    df['ny'] = ckpt['N'][:,1].tolist()
    
    # CE values
    df['Hxy'] = ckpt['M'][:,0].tolist()
    df['Hyx'] = ckpt['M'][:,1].tolist()
    
    df_ += [df]

df = pd.concat(df_, ignore_index=True)
print(df.shape)
df.head()

#### Creating Add'l Regression Model Variables

In [ ]:
# df['age_dif'] = df['x_age'] - df['y_age']
# df['pol_dif'] = df['x_politics'] - df['y_politics']
# df['status_dif'] = df['x_i_think_my_status'] - df['x_i_think_your_status']
# df['same_gender'] = (df['x_sex'] == df['y_sex']).astype(int)
# df['same_race'] = (df['x_race'] == df['y_race']).astype(int)
# df['same_edu'] = (df['x_edu'] == df['y_edu']).astype(int)
# df['t_delta'] = df['y_turn_id'] - df['x_turn_id']

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].mean()

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].median()

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].std()

## Save data

In [ ]:
df.to_csv(
    OUTPUT_NAME,
    index=False, 
    encoding='utf-8',
    sep='\t'
)